In [140]:
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [141]:
path = "datasets/aog_test"
names_list = ['components.csv', 'fleet_info.csv','maintenance_events.csv', 'sensor_readings.csv']
dfs = {}
for name in names_list:
    dfs[name] = pd.read_csv(os.path.join(path, name))

components_df, fleet_info_df, maintenance_events_df, sensor_readings_df = dfs['components.csv'], dfs['fleet_info.csv'], dfs['maintenance_events.csv'], dfs['sensor_readings.csv']

In [142]:
components_df.head()

# how many components?
# filter status = active?
# how many components per aircraft?
# are components unique?

,component_id,aircraft_id,component_type,installation_date,status
0,C001,AC014,actuator,2023-08-06,active
1,C002,AC012,bearing,2023-07-11,active
2,C003,AC006,actuator,2023-02-05,replaced
3,C004,AC007,hydraulic_pump,2023-08-04,replaced
4,C005,AC015,bearing,2023-06-07,active


In [143]:
print('total components:', len(components_df))
print('component_id unique:', components_df['component_id'].is_unique)
print(components_df['status'].value_counts())

active_components_df = components_df[components_df['status'] == 'active']
print('active components:', len(active_components_df))

components_df.groupby('aircraft_id').agg(n_components=('component_id', 'count'),
                                         n_unique=('component_id', 'nunique'))

total components: 50
component_id unique: True
status
active      39
replaced    11
Name: count, dtype: int64
active components: 39


,n_components,n_unique
aircraft_id,,
AC001,1,1
AC002,5,5
AC004,2,2
AC005,3,3
AC006,3,3
AC007,5,5
AC008,2,2
AC009,2,2
AC010,1,1


In [144]:
fleet_info_df.head()
# how many aircraft models?
# mean manufacture_year?
# when is total flight hours calculated?

,aircraft_id,aircraft_model,manufacture_year,base_airport,total_flight_hours
0,AC001,E190,2009,LIM,52814
1,AC002,B737-800,2017,SCL,45980
2,AC003,B737-800,2015,SCL,46680
3,AC004,A320,2005,SCL,32501
4,AC005,A320,2010,SCL,54382


In [145]:
print('aircraft:', fleet_info_df['aircraft_id'].nunique())
print(fleet_info_df['aircraft_model'].value_counts())
print('mean manufacture_year:', fleet_info_df['manufacture_year'].mean())

aircraft: 20
aircraft_model
B737-800    8
E190        6
A320        6
Name: count, dtype: int64
mean manufacture_year: 2011.25


In [146]:
maintenance_events_df.head()
# component_id is the key, not aircraft
# how many maintenance_type -> i see different types
# does the action matter?
# check downtime_hours depending on maintenance_type

,event_id,component_id,flight_cycle,maintenance_type,action,downtime_hours,aircraft_id,event_date
0,1,C017,21.0,scheduled,cleaning,2.0,AC001,2024-01-08 04:00
1,2,C008,97.0,Scheduled,inspection,1.0,AC007,2024-02-02 12:00
2,3,C008,11.0,scheduled,adjustment,5.0,AC007,2024-01-04 20:00
3,4,C014,100.0,Scheduled,lubrication,2.0,AC005,2024-02-03 12:00
4,5,C012,98.0,UNSCHEDULED,replacement,30.0,AC018,2024-02-02 20:00


In [147]:
print(maintenance_events_df['maintenance_type'].unique())

# same type written in different cases (scheduled / Scheduled / UNSCHEDULED)
maintenance_events_df['maintenance_type'] = maintenance_events_df['maintenance_type'].str.strip().str.lower()

maintenance_events_df.groupby(['maintenance_type', 'action'])['downtime_hours'].agg(['count', 'mean', 'median', 'max'])

<StringArray>
[   'scheduled',    'Scheduled', 'UNSCHEDULED ',    'SCHEDULED',
  'Unscheduled',  'unscheduled']
Length: 6, dtype: str


count       mean  median   max
maintenance_type action                                      
scheduled        adjustment       48   3.333333     3.0   5.0
                 cleaning         44   3.090909     3.0   5.0
                 inspection       54   2.666667     3.0   5.0
                 lubrication      42   3.023810     3.0   5.0
unscheduled      major_repair      7  28.714286    31.0  50.0
                 replacement       4  51.000000    52.5  69.0

In [148]:
sensor_readings_df.head()
# flight_cycle is proxy for timestamp?
# temp, vibrations and pressure are on different scales, need to normalize -> maybe z-score?
# components should have different sensor readings

,reading_id,component_id,flight_cycle,timestamp,temperature_celsius,vibration_mm_s,pressure_psi
0,1,C001,1,2024-01-01 08:14:00,59.756919,2.815316,1943.090329
1,2,C001,2,2024-01-01 16:48:00,58.976611,2.752847,1958.279957
2,3,C001,3,2024-01-02 00:35:00,60.483151,3.086081,2052.898299
3,4,C001,4,2024-01-02 08:49:00,61.875312,3.336544,2046.006625
4,5,C001,5,2024-01-02 16:37:00,55.742453,3.294884,2013.584800


In [149]:
sensor_readings_df[sensor_readings_df['component_id'] == 'C001'][['component_id','flight_cycle','timestamp']]

,component_id,flight_cycle,timestamp
0,C001,1,2024-01-01 08:14:00
1,C001,2,2024-01-01 16:48:00
2,C001,3,2024-01-02 00:35:00
3,C001,4,2024-01-02 08:49:00
4,C001,5,2024-01-02 16:37:00
...,...,...,...
96,C001,96,2024-02-02 00:03:00
97,C001,97,2024-02-02 08:47:00
98,C001,98,2024-02-02 16:20:00
99,C001,99,2024-02-03 00:04:00


In [150]:
sensor_readings_df.groupby('component_id').agg({'flight_cycle':'count'}).query('flight_cycle > 100')
#  Son 100 ciclos por componente -> more cycles than stated

,flight_cycle
component_id,
C001,101
C010,101
C012,101
C015,102
C018,102
C020,102
C024,101
C036,101
C040,101


In [151]:
# flight_cycle as proxy for timestamp: timestamp should always increase with flight_cycle
sensor_readings_df['timestamp'] = pd.to_datetime(sensor_readings_df['timestamp'])
sorted_df = sensor_readings_df.sort_values(['component_id', 'flight_cycle'])
print('timestamp increases with flight_cycle:',
      sorted_df.groupby('component_id')['timestamp'].apply(lambda x: x.is_monotonic_increasing).all())
print('repeated (component_id, flight_cycle):', sensor_readings_df.duplicated(['component_id', 'flight_cycle']).sum())

timestamp increases with flight_cycle: True
repeated (component_id, flight_cycle): 15


In [152]:
# different scales -> normalize with z-score per component_type
sensor_cols = ['temperature_celsius', 'vibration_mm_s', 'pressure_psi']
sensor_readings_df['component_type'] = sensor_readings_df['component_id'].map(components_df.set_index('component_id')['component_type'])
display(sensor_readings_df.groupby('component_type')[sensor_cols].describe())

grouped = sensor_readings_df.groupby('component_type')
for col in sensor_cols:
    sensor_readings_df[col + '_z'] = (sensor_readings_df[col] - grouped[col].transform('mean')) / grouped[col].transform('std')

temperature_celsius                                   \
                             count       mean        std        min   
component_type                                                        
actuator                    1079.0  61.187218  28.669869  53.209905   
bearing                     1279.0  85.250641   2.988046  74.740945   
filter                      1184.0  45.374767   1.905669  40.673972   
hydraulic_pump               982.0  77.086794  41.763360  68.146661   
valve                        391.0  55.097684   1.632688  49.808737   

                                                            vibration_mm_s  \
                      25%        50%        75%         max          count   
component_type                                                               
actuator        58.973785  60.113463  61.409700  999.000000         1084.0   
bearing         83.316033  85.112513  86.932533  101.837071         1275.0   
filter          44.218823  45.175450  46.266102   55.549351         1179.0   
hydraulic_pump  73.378986  75.001267  76.529385  999.000000          985.0   
valve           53.980145  55.102071  56.193638   59.749373          392.0   

                          ...                      pressure_psi               \
                    mean  ...       75%        max        count         mean   
component_type            ...                                                  
actuator        3.094732  ...  3.179535   6.834403       1073.0  1993.436933   
bearing         6.124110  ...  6.381685  12.878718       1283.0    24.962018   
filter          1.566647  ...  1.612637   3.407650       1187.0   298.176671   
hydraulic_pump  4.085278  ...  4.256085   9.025809        981.0  2994.834079   
valve           1.998800  ...  2.105224   2.475612        391.0  1499.381452   

                                                                               \
                      std          min          25%          50%          75%   
component_type                                                                  
actuator        77.530138    -1.000000  1968.240619  1998.978473  2027.243200   
bearing          0.562485    21.706988    24.646420    24.973732    25.337289   
filter          14.321641    -1.000000   294.949375   299.332264   303.343228   
hydraulic_pump  70.474135  2674.864254  2953.678215  2998.447258  3041.112432   
valve           28.766490  1409.656498  1480.614935  1500.110661  1518.676347   

                             
                        max  
component_type               
actuator        2122.430976  
bearing           26.558740  
filter           318.790238  
hydraulic_pump  3177.153871  
valve           1575.889453  

[5 rows x 24 columns]

In [153]:
# nulls and duplicates in every table
for name, df in dfs.items():
    print(name)
    print(df.isna().sum()[df.isna().sum() > 0])
    print('duplicates:', df.duplicated().sum(), '\n')

# 100 nulls en sensor_readings -> no sense inputing average, we should use last flight_cycle value

components.csv
Series([], dtype: int64)
duplicates: 0 

fleet_info.csv
Series([], dtype: int64)
duplicates: 0 

maintenance_events.csv
flight_cycle    2
dtype: int64
duplicates: 0 

sensor_readings.csv
temperature_celsius      100
vibration_mm_s           100
pressure_psi             100
temperature_celsius_z    100
vibration_mm_s_z         100
pressure_psi_z           100
dtype: int64
duplicates: 0 



In [154]:
# extreme values: |z| > 3
def count_outliers(df, cols, threshold=3):
    z = (df[cols] - df[cols].mean()) / df[cols].std()
    return (z.abs() > threshold).sum()

print(count_outliers(fleet_info_df, ['manufacture_year', 'total_flight_hours']))
print(count_outliers(maintenance_events_df, ['downtime_hours']))
print(count_outliers(sensor_readings_df, sensor_cols))

manufacture_year      0
total_flight_hours    0
dtype: int64
downtime_hours    6
dtype: int64
temperature_celsius     3
vibration_mm_s         20
pressure_psi            0
dtype: int64


## Data cleaning

In [155]:
# duplicated readings: same component_id and flight_cycle, only reading_id changes
sensor_readings_df = sensor_readings_df.drop_duplicates(subset=['component_id', 'flight_cycle'])
sensor_readings_df = sensor_readings_df.sort_values(['component_id', 'flight_cycle'])
print(sensor_readings_df.groupby('component_id')['flight_cycle'].count().value_counts())

flight_cycle
100    50
Name: count, dtype: int64


In [ ]:
# placeholder values (temperature 999, pressure -1) are sensor errors, not readings -> treat as nulls
print('placeholders:', sensor_readings_df[sensor_cols].isin([999, -1]).sum().sum())
sensor_readings_df[sensor_cols] = sensor_readings_df[sensor_cols].replace({999: np.nan, -1: np.nan})

# nulls: use the last flight_cycle value of the same component
sensor_readings_df[sensor_cols] = sensor_readings_df.groupby('component_id')[sensor_cols].ffill()
# first flight_cycle has no previous value -> use the next one
sensor_readings_df[sensor_cols] = sensor_readings_df.groupby('component_id')[sensor_cols].bfill()
print(sensor_readings_df[sensor_cols].isna().sum())

In [157]:
# z-score normalization per component_type
grouped = sensor_readings_df.groupby('component_type')
for col in sensor_cols:
    sensor_readings_df[col + '_z'] = (sensor_readings_df[col] - grouped[col].transform('mean')) / grouped[col].transform('std')

sensor_readings_df.head()

,reading_id,component_id,flight_cycle,timestamp,temperature_celsius,vibration_mm_s,pressure_psi,component_type,temperature_celsius_z,vibration_mm_s_z,pressure_psi_z
0,1,C001,1,2024-01-01 08:14:00,59.756919,2.815316,1943.090329,actuator,-0.050018,-0.524133,-0.657016
1,2,C001,2,2024-01-01 16:48:00,58.976611,2.752847,1958.279957,actuator,-0.077494,-0.640339,-0.459297
2,3,C001,3,2024-01-02 00:35:00,60.483151,3.086081,2052.898299,actuator,-0.024446,-0.020455,0.772326
3,4,C001,4,2024-01-02 08:49:00,61.875312,3.336544,2046.006625,actuator,0.024575,0.445457,0.682618
4,5,C001,5,2024-01-02 16:37:00,55.742453,3.294884,2013.584800,actuator,-0.191374,0.367962,0.260592


## Target: will_fail_within_30_cycle

In [158]:
# first unscheduled failure cycle per component
failures = (maintenance_events_df[maintenance_events_df['maintenance_type'] == 'unscheduled']
            .groupby('component_id')['flight_cycle'].min().rename('failure_cycle'))
sensor_readings_df = sensor_readings_df.merge(failures, on='component_id', how='left')

# 1 if the failure happens in this cycle or the next 30
cycles_to_failure = sensor_readings_df['failure_cycle'] - sensor_readings_df['flight_cycle']
sensor_readings_df['will_fail_within_30_cycle'] = cycles_to_failure.between(0, 30).astype(int)

In [159]:
# censor: no failure seen, but the 30-cycle window goes past the last observed cycle -> label unknown, drop
last_cycle = sensor_readings_df.groupby('component_id')['flight_cycle'].transform('max')
censored = (sensor_readings_df['will_fail_within_30_cycle'] == 0) & (sensor_readings_df['flight_cycle'] + 30 > last_cycle)
model_df = sensor_readings_df[~censored]
print('censored rows dropped:', censored.sum())

# class balance
print(model_df['will_fail_within_30_cycle'].value_counts())
print(model_df['will_fail_within_30_cycle'].value_counts(normalize=True))

censored rows dropped: 1252
will_fail_within_30_cycle
0    3407
1     341
Name: count, dtype: int64
will_fail_within_30_cycle
0    0.909018
1    0.090982
Name: proportion, dtype: float64


## Feature engineering

In [160]:
def add_sensor_features(df, baseline_components):
    """Sensor features that only use cycles up to t.
    The healthy baseline (first 10 cycles, failures start after cycle 60) is taken per component_type,
    only from baseline_components, so validation can build it from the training components alone."""
    df = df.copy()
    component_type = df['component_id'].map(components_df.set_index('component_id')['component_type'])

    # z-score against the healthy baseline
    healthy = df['component_id'].isin(baseline_components) & (df['flight_cycle'] <= 10)
    baseline = df[healthy].groupby(component_type[healthy])[sensor_cols].agg(['mean', 'std'])
    for col in sensor_cols:
        mean = component_type.map(baseline[(col, 'mean')])
        std = component_type.map(baseline[(col, 'std')])
        df[col + '_z'] = (df[col] - mean) / std
    z_cols = [col + '_z' for col in sensor_cols]

    # rolling mean and std over the last 5 and 10 cycles (rows are sorted by flight_cycle)
    grouped = df.groupby('component_id')
    for col in z_cols:
        for w in [5, 10]:
            df[f'{col}_mean_{w}'] = grouped[col].transform(lambda x: x.rolling(w).mean())
            df[f'{col}_std_{w}'] = grouped[col].transform(lambda x: x.rolling(w).std())

    # health score: mean absolute z-score of the three sensors at cycle t
    df['health_score'] = df[z_cols].abs().mean(axis=1)
    return df

In [161]:
# baseline from all components: only for exploration and the leakage check
# validation rebuilds these features per fold with the training components only
sensor_readings_df = add_sensor_features(sensor_readings_df, sensor_readings_df['component_id'].unique())

In [162]:
# context features (status is left out: replaced = failed, it leaks the target)
aircraft_id = sensor_readings_df['component_id'].map(components_df.set_index('component_id')['aircraft_id'])
sensor_readings_df['aircraft_model'] = aircraft_id.map(fleet_info_df.set_index('aircraft_id')['aircraft_model'])
sensor_readings_df['base_airport'] = aircraft_id.map(fleet_info_df.set_index('aircraft_id')['base_airport'])

sensor_readings_df = pd.get_dummies(sensor_readings_df, columns=['component_type', 'aircraft_model', 'base_airport'], dtype=int)

In [163]:
# modelling table: drop censored rows and the baseline cycles (1-10)
# failure_cycle is dropped because it is the answer the target is built from
model_df = sensor_readings_df[~censored & (sensor_readings_df['flight_cycle'] > 10)].drop(columns='failure_cycle')
print(model_df.shape)
print(model_df['will_fail_within_30_cycle'].value_counts())
model_df.groupby('will_fail_within_30_cycle')['health_score'].describe()

(3248, 35)
will_fail_within_30_cycle
0    2907
1     341
Name: count, dtype: int64


,count,mean,std,min,25%,50%,75%,max
will_fail_within_30_cycle,,,,,,,,
0,2907.0,0.986964,4.839016,0.052688,0.574611,0.782886,1.040027,172.599545
1,341.0,3.479010,2.495748,0.174507,1.191965,2.810708,5.477417,10.238964


In [164]:
# leakage check: AUC of each feature alone against the target
# strength = distance from 0.5 (AUC 0.1 is as strong as 0.9, just inverted); > 0.9 is suspicious
from sklearn.metrics import roc_auc_score

target = 'will_fail_within_30_cycle'
feature_cols = model_df.select_dtypes('number').columns.drop(target)

leak_check = pd.Series({col: roc_auc_score(model_df[target], model_df[col]) for col in feature_cols}, name='auc').to_frame()
leak_check['strength'] = (leak_check['auc'] - 0.5).abs() + 0.5
leak_check['flag'] = leak_check['strength'] > 0.9
leak_check.sort_values('strength', ascending=False)

,auc,strength,flag
flight_cycle,0.939876,0.939876,True
pressure_psi_z_mean_5,0.115971,0.884029,False
temperature_celsius_z_mean_5,0.879595,0.879595,False
temperature_celsius_z_mean_10,0.876851,0.876851,False
pressure_psi_z_mean_10,0.123465,0.876535,False
health_score,0.868181,0.868181,False
vibration_mm_s_z_mean_5,0.862644,0.862644,False
vibration_mm_s_z_mean_10,0.862515,0.862515,False
vibration_mm_s_z,0.856996,0.856996,False
pressure_psi_z,0.145442,0.854558,False


## Validation

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

# flight_cycle leaks (failures only happen after cycle 60), timestamp tracks it; ids are not features
X = model_df.drop(columns=[target, 'flight_cycle', 'timestamp', 'reading_id', 'component_id'])
# aircraft_model and base_airport encodings are left out of the models
X = X.loc[:, ~X.columns.str.startswith(('aircraft_model_', 'base_airport_'))]
y = model_df[target]
groups = model_df['component_id']

# folds split by component (never rows at random), keeping the % of positives similar across folds
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

scores = []
for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups)):
    # rebuild sensor features with a baseline from the training components only (no test leakage)
    train_components = groups.iloc[train_idx].unique()
    X_fold = add_sensor_features(sensor_readings_df, train_components).loc[X.index, X.columns]

    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced'))
    model.fit(X_fold.iloc[train_idx], y.iloc[train_idx])
    pred = model.predict_proba(X_fold.iloc[test_idx])[:, 1]

    # PR-AUC: a random model scores the positive rate, not 0.5
    pr_auc = average_precision_score(y.iloc[test_idx], pred)
    scores.append(pr_auc)
    print(f'fold {fold}: test components={groups.iloc[test_idx].nunique()}, '
          f'positive rate={y.iloc[test_idx].mean():.3f}, PR-AUC={pr_auc:.3f}')

print(f'mean PR-AUC: {np.mean(scores):.3f} +/- {np.std(scores):.3f}')

## Modeling

In [166]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import precision_score, recall_score
from lightgbm import LGBMClassifier

def make_logreg():
    # no class_weight, so the output stays a real probability of failure
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

def forward_auc_selection(X, y, groups, min_gain=0.005):
    """Add one feature at a time, keeping the one that raises ROC AUC the most
    (grouped CV on the training components). Stop when the best gain is below min_gain."""
    inner_cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
    selected, best_auc = [], 0.5
    while len(selected) < X.shape[1]:
        aucs = {col: cross_val_score(make_logreg(), X[selected + [col]], y, groups=groups,
                                     cv=inner_cv, scoring='roc_auc').mean()
                for col in X.columns.drop(selected)}
        col = max(aucs, key=aucs.get)
        if aucs[col] - best_auc < min_gain:
            break
        selected.append(col)
        best_auc = aucs[col]
    return selected

In [167]:
# out-of-fold predictions: every row is predicted by models that never saw its component
oof = pd.DataFrame(index=X.index, columns=['random', 'logreg', 'lightgbm'], dtype=float)
fold_results, importances = [], []

for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups)):
    train_components = groups.iloc[train_idx].unique()
    X_fold = add_sensor_features(sensor_readings_df, train_components).loc[X.index, X.columns]
    X_train, X_test = X_fold.iloc[train_idx], X_fold.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # 1. random model: always predicts the training positive rate
    oof.loc[X_test.index, 'random'] = y_train.mean()

    # 2. logistic regression with forward AUC selection (done on training components only)
    features = forward_auc_selection(X_train, y_train, groups.iloc[train_idx])
    logreg = make_logreg().fit(X_train[features], y_train)
    oof.loc[X_test.index, 'logreg'] = logreg.predict_proba(X_test[features])[:, 1]
    print(f'fold {fold} logreg features: {features}')

    # 3. lightgbm with every feature
    lgbm = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15,
                          importance_type='gain', random_state=42, verbose=-1)
    lgbm.fit(X_train, y_train)
    oof.loc[X_test.index, 'lightgbm'] = lgbm.predict_proba(X_test)[:, 1]
    importances.append(pd.Series(lgbm.feature_importances_, index=X.columns))

    fold_results.append({'fold': fold, 'positive_rate': y_test.mean(),
                         **{m: average_precision_score(y_test, oof.loc[X_test.index, m]) for m in oof.columns}})

fold 0 logreg features: ['temperature_celsius_z_mean_5', 'component_type_valve', 'temperature_celsius_z_std_10', 'pressure_psi_z_mean_10']
fold 1 logreg features: ['pressure_psi_z_mean_5', 'vibration_mm_s_z_std_10', 'component_type_valve']
fold 2 logreg features: ['temperature_celsius_z_mean_10', 'vibration_mm_s_z_std_5', 'component_type_valve']
fold 3 logreg features: ['pressure_psi_z_mean_5', 'component_type_bearing', 'base_airport_SCL', 'component_type_valve']
fold 4 logreg features: ['temperature_celsius_z_mean_5', 'temperature_celsius_z_std_10', 'pressure_psi_z_mean_5', 'pressure_psi']


In [168]:
# PR-AUC per fold (random model = positive rate)
results = pd.DataFrame(fold_results).set_index('fold')
display(results.round(3))
display(results.agg(['mean', 'std']).round(3))

,positive_rate,random,logreg,lightgbm
fold,,,,
0,0.095,0.095,0.759,0.818
1,0.095,0.095,0.772,0.808
2,0.096,0.096,0.368,0.810
3,0.141,0.141,0.269,0.788
4,0.098,0.098,0.766,0.791


,positive_rate,random,logreg,lightgbm
mean,0.105,0.105,0.587,0.803
std,0.020,0.020,0.248,0.013


In [169]:
# lightgbm feature significance: share of total gain, averaged over folds
importance = pd.concat(importances, axis=1).mean(axis=1)
(importance / importance.sum()).sort_values(ascending=False).round(3)

vibration_mm_s_z_mean_5          0.266
pressure_psi_z_mean_5            0.152
temperature_celsius_z_mean_5     0.118
vibration_mm_s_z                 0.072
health_score                     0.039
temperature_celsius_z_mean_10    0.038
temperature_celsius_z_std_10     0.037
vibration_mm_s_z_std_10          0.037
pressure_psi_z_mean_10           0.026
vibration_mm_s_z_mean_10         0.026
vibration_mm_s_z_std_5           0.023
pressure_psi_z_std_10            0.023
pressure_psi_z_std_5             0.017
pressure_psi                     0.015
temperature_celsius_z_std_5      0.015
vibration_mm_s                   0.015
pressure_psi_z                   0.014
temperature_celsius              0.013
temperature_celsius_z            0.012
aircraft_model_E190              0.010
aircraft_model_A320              0.008
base_airport_LIM                 0.007
aircraft_model_B737-800          0.005
base_airport_SCL                 0.004
base_airport_BOG                 0.003
component_type_actuator  

In [170]:
# precision and recall at different thresholds (out-of-fold predictions)
rows = []
for threshold in np.arange(0.1, 1.0, 0.1).round(1):
    for model_name in ['logreg', 'lightgbm']:
        pred = (oof[model_name] >= threshold).astype(int)
        rows.append({'threshold': threshold, 'model': model_name,
                     'precision': precision_score(y, pred, zero_division=0),
                     'recall': recall_score(y, pred),
                     'flagged': pred.sum()})

pd.DataFrame(rows).pivot(index='threshold', columns='model', values=['precision', 'recall', 'flagged']).round(3)

precision          recall         flagged       
model      lightgbm logreg lightgbm logreg lightgbm logreg
threshold                                                 
0.1           0.876  0.382    0.686  0.569    267.0  508.0
0.2           0.941  0.889    0.660  0.469    239.0  180.0
0.3           0.962  0.898    0.660  0.413    234.0  157.0
0.4           0.965  0.887    0.651  0.367    230.0  141.0
0.5           0.969  0.876    0.645  0.331    227.0  129.0
0.6           0.973  0.862    0.636  0.293    223.0  116.0
0.7           0.982  0.835    0.633  0.238    220.0   97.0
0.8           0.982  0.807    0.633  0.196    220.0   83.0
0.9           0.981  0.733    0.622  0.129    216.0   60.0

In [ ]:
# precision-recall curve per model (out-of-fold predictions); the area under each curve is its PR-AUC
from sklearn.metrics import precision_recall_curve

fig, ax = plt.subplots(figsize=(7, 5))
for model_name, color in {'lightgbm': '#2a78d6', 'logreg': '#eb6834'}.items():
    precision, recall, _ = precision_recall_curve(y, oof[model_name])
    pr_auc = average_precision_score(y, oof[model_name])
    ax.plot(recall, precision, color=color, linewidth=2, label=f'{model_name} (PR-AUC = {pr_auc:.2f})')

# random model: precision is the positive rate at every recall
ax.axhline(y.mean(), color='#8a8a85', linestyle='--', linewidth=1.5, label=f'random (PR-AUC = {y.mean():.2f})')

ax.set(xlabel='Recall', ylabel='Precision', title='Precision-recall curve (out-of-fold predictions)',
       xlim=(0, 1), ylim=(0, 1.02))
ax.grid(alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False, loc='lower left', bbox_to_anchor=(0.1, 0.2))
plt.show()

## Threshold trade-off: unnecessary inspections vs unscheduled failures

In [ ]:
# aircraft-level trade-off: unnecessary inspections vs unscheduled failures
# a failure is avoided only if the first alert comes early enough to act (readings are 8h apart: 10 cycles ~ 3 days, 20 ~ 1 week)
lead_requirements = [10, 20]

eval_df = pd.DataFrame({'component_id': groups, 'y': y})
eval_df['aircraft_id'] = eval_df['component_id'].map(components_df.set_index('component_id')['aircraft_id'])
eval_df['cycles_to_failure'] = sensor_readings_df.loc[eval_df.index, 'failure_cycle'] - model_df['flight_cycle']
failing = eval_df[eval_df['y'] == 1]
# unnecessary inspection = alert on a component that is not about to fail (readings after a failure are left out)
healthy = eval_df['cycles_to_failure'].isna() | (eval_df['cycles_to_failure'] > 30)

rows = []
for model_name in ['logreg', 'lightgbm']:
    for threshold in np.arange(0.05, 1.0, 0.05).round(2):
        alert = oof[model_name] >= threshold
        first_alert = failing[alert[failing.index]].groupby('component_id')['cycles_to_failure'].max()
        row = {'model': model_name, 'threshold': threshold,
               'precision': precision_score(y, alert, zero_division=0),
               'recall': recall_score(y, alert),
               'median_warning_cycles': first_alert.median(),
               'aircraft_unnecessary_inspection': eval_df.loc[alert & healthy, 'aircraft_id'].nunique()}
        for lead in lead_requirements:
            warned = first_alert[first_alert >= lead].index
            row[f'aircraft_failure_{lead}_cycles'] = failing.loc[~failing['component_id'].isin(warned), 'aircraft_id'].nunique()
        rows.append(row)

tradeoff = pd.DataFrame(rows)
print(f"aircraft: {eval_df['aircraft_id'].nunique()} | with a failure: {failing['aircraft_id'].nunique()} | failing components: {failing['component_id'].nunique()}")
tradeoff.round(3)

In [ ]:
from matplotlib.ticker import MaxNLocator

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, model_name in zip(axes, ['lightgbm', 'logreg']):
    t = tradeoff[tradeoff['model'] == model_name]
    ax.plot(t['threshold'], t['aircraft_unnecessary_inspection'], color='#2a78d6', linewidth=2, marker='o', markersize=4,
            label='inspected unnecessarily')
    ax.plot(t['threshold'], t['aircraft_failure_10_cycles'], color='#eb6834', linewidth=2, marker='s', markersize=4,
            label='unscheduled failure (alert needed 10+ cycles ahead)')
    ax.plot(t['threshold'], t['aircraft_failure_20_cycles'], color='#eb6834', linewidth=2, linestyle='--', marker='s', markersize=4,
            markerfacecolor='white', label='unscheduled failure (alert needed 20+ cycles ahead)')
    ax.set(title=model_name, xlabel='Threshold', xlim=(0, 1))
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_ylabel(f"Aircraft (of {eval_df['aircraft_id'].nunique()})")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, frameon=False)
fig.suptitle('Unnecessary inspections vs unscheduled failures by threshold (out-of-fold predictions)')
plt.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()